In [4]:
import cloudscraper
from bs4 import BeautifulSoup
import csv

def scrape_ischool_precise():
    url = "https://www.ischool.berkeley.edu/people?role=122&faculty_type=74"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print("Fetching page...")
    response = scraper.get(url)
    
    if response.status_code != 200:
        print(f"Failed to load page. Status: {response.status_code}")
        return []

    print("Parsing data...")
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Get all rows on the page
    rows = soup.find_all('div', class_='views-row')
    results = []

    for row in rows:
        # 🎯 THE FILTER: Does this row have a 'fullname' class? 
        # If not, it's a news article or event, so skip it.
        name_wrapper = row.find('div', class_='views-field-field-profile-fullname')
        if not name_wrapper:
            continue
            
        # 1. Extract Name
        # Based on your image: div -> h2 -> a
        name = name_wrapper.get_text(strip=True)
        
        # 2. Extract Position
        # Based on your image: class="views-field-field-profile-subtitle"
        subtitle_wrapper = row.find('div', class_='views-field-field-profile-subtitle')
        position = subtitle_wrapper.get_text(strip=True) if subtitle_wrapper else "N/A"
        
        # 3. Extract Email
        # Based on your image: class="views-field-field-profile-email"
        email_wrapper = row.find('div', class_='views-field-field-profile-email')
        if email_wrapper:
            # We can grab the text straight from the <a> tag inside it
            email_tag = email_wrapper.find('a')
            email = email_tag.get_text(strip=True) if email_tag else "N/A"
        else:
            email = "N/A"

        results.append({
            'Name': name,
            'Position': position,
            'Email': email
        })

    return results

# Run the script
faculty_data = scrape_ischool_precise()

if faculty_data:
    filename = 'ischool_faculty_final.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(faculty_data)
    
    print(f"\nSuccess! Cleaned and extracted {len(faculty_data)} profiles.")
    print(f"Data saved to {filename}\n")
    
    print("--- Top 3 Results ---")
    for person in faculty_data[:3]:
        print(person)
else:
    print("No profiles found.")

Fetching page...
Parsing data...

Success! Cleaned and extracted 24 profiles.
Data saved to ischool_faculty_final.csv

--- Top 3 Results ---
{'Name': 'Alissa “Dr J” Abdullah', 'Position': 'Lecturer', 'Email': 'dralissajay@ischool.berkeley.edu'}
{'Name': 'Elliott Adams', 'Position': 'Lecturer', 'Email': 'N/A'}
{'Name': 'Mak Ahmad', 'Position': 'Lecturer', 'Email': 'mak@ischool.berkeley.edu'}


In [6]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv

def scrape_sociology_table():
    url = "https://sociology.berkeley.edu/people/visiting-faculty"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print("Fetching sociology faculty table...")
    response = scraper.get(url)
    
    if response.status_code != 200:
        print(f"Failed. Status code: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Target the table body
    tbody = soup.find('tbody')
    if not tbody:
        print("Could not find the table body. The page structure may have changed.")
        return []
        
    # Get all rows in the table
    rows = tbody.find_all('tr')
    results = []

    for row in rows:
        # 1 & 2. Extract Name and Position from the first column
        name_td = row.find('td', class_=re.compile(r'views-field-name'))
        if not name_td:
            continue
            
        # Name is inside the <a> tag
        name_tag = name_td.find('a')
        name = name_tag.get_text(strip=True) if name_tag else "N/A"
        
        # Position is inside the <div> just below the <a> tag
        position_div = name_td.find('div')
        position = position_div.get_text(strip=True) if position_div else "N/A"

        # 3. Extract Email from the second column (Contact)
        contact_td = row.find('td', class_=re.compile(r'views-field-mail'))
        if contact_td:
            contact_text = contact_td.get_text(" ", strip=True)
            email_match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', contact_text)
            email = email_match.group(0) if email_match else "N/A"
        else:
            email = "N/A"

        results.append({
            'Name': name,
            'Position': position,
            'Email': email
        })

    return results

# Run it and export
data = scrape_sociology_table()

if data:
    filename = 'berkeley_sociology_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\nSuccess! Extracted {len(data)} profiles. Saved to {filename}.")
    print("\n--- Top 3 Results ---")
    for row in data[:3]:
        print(row)

Fetching sociology faculty table...

Success! Extracted 12 profiles. Saved to berkeley_sociology_faculty.csv.

--- Top 3 Results ---
{'Name': 'Jill A. Bakehorn', 'Position': 'Continuing Lecturer', 'Email': 'jabakehorn@berkeley.edu'}
{'Name': 'Laleh Behbehanian', 'Position': 'Continuing Lecturer', 'Email': 'lalehb@berkeley.edu'}
{'Name': 'Fatmir Haskaj', 'Position': 'N/A', 'Email': 'haskaj@berkeley.edu'}


In [8]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_polisci_faculty():
    base_url = "https://polisci.berkeley.edu"
    main_url = f"{base_url}/people/faculty"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    articles = soup.find_all('article', typeof="schema:Person")
    
    faculty_list = []

    for article in articles:
        # Profile Link
        link_tag = article.find('a', href=True)
        if not link_tag:
            continue
        profile_path = link_tag['href']
        
        # Name
        name_div = article.find('div', class_=re.compile(r'field--name-realname'))
        name = name_div.get_text(strip=True) if name_div else "N/A"
        
        # Position
        title_div = article.find('div', class_=re.compile(r'field--name-field-user-title'))
        position = title_div.get_text(strip=True) if title_div else "N/A"
        
        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} faculty members. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive (UPDATED) ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                # Parse the individual profile page
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # 🎯 TARGETED FIX: Find the <a> tag with class "mailto" or href starting with "mailto:"
                email_tag = prof_soup.find('a', href=re.compile(r'^mailto:'))
                
                if email_tag:
                    # Extract the email and clean off the "mailto:" prefix
                    raw_email = email_tag['href'].replace('mailto:', '')
                    # Sometimes emails have "?subject=..." attached, so we split at "?" and take the first part
                    person['Email'] = raw_email.split('?')[0].strip()
                else:
                    person['Email'] = "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # Politeness delay
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_polisci_faculty()

if data:
    filename = 'berkeley_polisci_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Deep scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://polisci.berkeley.edu/people/faculty)
Found 49 faculty members. Starting Phase 2...

Scraping 1/49: Vinod Aggarwal...
Scraping 2/49: Christopher Ansell...
Scraping 3/49: Sarah Anzia...
Scraping 4/49: Leonardo Arriola...
Scraping 5/49: Kirk Bansak...
Scraping 6/49: Mark Bevir...
Scraping 7/49: Terri Bimes...
Scraping 8/49: Henry Brady...
Scraping 9/49: David Broockman...
Scraping 10/49: Ryan Brutger...
Scraping 11/49: Jennifer Bussell...
Scraping 12/49: Daniela Cammack...
Scraping 13/49: Pradeep K Chhibber...
Scraping 14/49: Amanda Clayton...
Scraping 15/49: Ernesto Dal Bo...
Scraping 16/49: Thad Dunning...
Scraping 17/49: Barry Eichengreen...
Scraping 18/49: M. Steven Fish...
Scraping 19/49: Sean Gailmard...
Scraping 20/49: Erin Hartman...
Scraping 21/49: Ron Hassner...
Scraping 22/49: Kinch Hoekstra...
Scraping 23/49: Susan Hyde...
Scraping 24/49: Desmond Jagmohan...
Scraping 25/49: Marika Landau-Wells...
Scraping 26/49: Daniel Lee...
Scrapi

In [9]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_media_studies():
    # Assuming this is the main directory URL based on your images
    base_url = "https://mediastudies.ugis.berkeley.edu"
    main_url = f"{base_url}/people/" 
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 🎯 TARGET 1: The article container from the first screenshot
    articles = soup.find_all('article', class_=re.compile(r'post-articles'))
    
    faculty_list = []

    for article in articles:
        # Find the content block
        content_div = article.find('div', class_=re.compile(r'ugis-post-content'))
        if not content_div:
            continue
            
        # 1. Get the Profile Link and Name
        # Looking for the first <a> tag inside the content block
        link_tag = content_div.find('a', href=True)
        if not link_tag:
            continue
            
        name = link_tag.get_text(strip=True)
        profile_path = link_tag['href']
        
        # 2. Get the Position
        # Targeting the exact <h3 class="aoc"> tag
        title_tag = content_div.find('h3', class_='aoc')
        position = title_tag.get_text(strip=True) if title_tag else "N/A"
        
        faculty_list.append({
            'Name': name,
            'Position': position,
            # If the href is already a full URL, urljoin handles it perfectly.
            'Profile_URL': urljoin(base_url, profile_path), 
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} faculty members. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # 🎯 TARGET 2: The contact-details div from the second screenshot
                contact_div = prof_soup.find('div', id='contact-details')
                
                if contact_div:
                    email_tag = contact_div.find('a', href=re.compile(r'^mailto:'))
                    if email_tag:
                        raw_email = email_tag['href'].replace('mailto:', '')
                        person['Email'] = raw_email.split('?')[0].strip()
                    else:
                        person['Email'] = "N/A"
                else:
                    # Fallback just in case the ID changes on some pages
                    email_tag = prof_soup.find('a', href=re.compile(r'^mailto:'))
                    person['Email'] = email_tag['href'].replace('mailto:', '').split('?')[0].strip() if email_tag else "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # Politeness delay
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_media_studies()

if data:
    filename = 'berkeley_media_studies.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://mediastudies.ugis.berkeley.edu/people/)
Found 8 faculty members. Starting Phase 2...

Scraping 1/8: Matthew Berry...
Scraping 2/8: Laura Demir...
Scraping 3/8: Josh Jackson...
Scraping 4/8: Rich Jaroslovsky...
Scraping 5/8: Ian Kivelin Davis...
Scraping 6/8: Chelsea Prieto...
Scraping 7/8: Meeta Rani Jha...
Scraping 8/8: Shannon Steen...

✅ Success! Scrape complete. Saved to berkeley_media_studies.csv.

--- Preview ---
{'Name': 'Matthew Berry', 'Position': 'Lecturer, Ph.D.  History, University of California, Berkeley', 'Email': 'matthewberry@berkeley.edu'}
{'Name': 'Laura Demir', 'Position': 'Media Studies', 'Email': 'demir@berkeley.edu'}
{'Name': 'Josh Jackson', 'Position': 'Assistant Director, Ph.D., U. of Wisconsin-Madison', 'Email': 'joshjackson@berkeley.edu'}


In [10]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_public_health():
    base_url = "https://publichealth.berkeley.edu"
    main_url = f"{base_url}/people"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 🎯 TARGET 1: The overarching <a> tag that wraps the whole profile card
    # Looking at your first screenshot, the card is an <a> tag with 'uk-link-toggle'
    card_links = soup.find_all('a', class_=re.compile(r'uk-link-toggle'))
    
    faculty_list = []

    for card in card_links:
        profile_path = card.get('href')
        if not profile_path:
            continue
            
        # The name and position are inside a div with class 'uk-link-text' inside the <a> tag
        text_block = card.find('div', class_=re.compile(r'uk-link-text'))
        
        if text_block:
            # Name is in a span with 'bph-text-serif'
            name_tag = text_block.find('span', class_=re.compile(r'bph-text-serif'))
            name = name_tag.get_text(strip=True) if name_tag else "N/A"
            
            # Position is in a span with 'uk-text-small'
            pos_tag = text_block.find('span', class_=re.compile(r'uk-text-small'))
            position = pos_tag.get_text(strip=True) if pos_tag else "N/A"
        else:
            continue

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} faculty members. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # 🎯 TARGET 2: The email link on the profile page
                # As seen in your second image, it's an <a> tag with 'uk-link-text' and a 'mailto:' href
                email_tag = prof_soup.find('a', class_=re.compile(r'uk-link-text'), href=re.compile(r'^mailto:'))
                
                if email_tag:
                    raw_email = email_tag['href'].replace('mailto:', '')
                    person['Email'] = raw_email.split('?')[0].strip()
                else:
                    person['Email'] = "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # 🛑 Politeness delay - critical for scraping ~200 profiles
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_public_health()

if data:
    filename = 'berkeley_public_health.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://publichealth.berkeley.edu/people)
Found 162 faculty members. Starting Phase 2...

Scraping 1/162: Barbara Abrams...
Scraping 2/162: Katrina Abuabara...
Scraping 3/162: Jennifer Ahern...
Scraping 4/162: Tomer Altman...
Scraping 5/162: Joshua Apte...
Scraping 6/162: Tomás Aragón...
Scraping 7/162: Colette Auerswald...
Scraping 8/162: Amin Azzam...
Scraping 9/162: John Balmes...
Scraping 10/162: Laura Balzer...
Scraping 11/162: Lisa Barcellos...
Scraping 12/162: Deborah Barnett...
Scraping 13/162: Michael Bates...
Scraping 14/162: Stefano Bertozzi...
Scraping 15/162: Gladys ​Block...
Scraping 16/162: Sabrina Boyce...
Scraping 17/162: Patrick Bradshaw...
Scraping 18/162: Jennifer Breckler...
Scraping 19/162: Amanda Brewster...
Scraping 20/162: Timothy Brown...
Scraping 21/162: Alex Budak...
Scraping 22/162: Xóchitl Castañeda...
Scraping 23/162: Anand Chokkalingam...
Scraping 24/162: John Colford...
Scraping 25/162: Jason Corburn...
Scraping 26/1

In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_haas_stealth_native():
    base_url = "https://haas.berkeley.edu"
    main_url = f"{base_url}/faculty/"
    
    print("Initializing stealth browser...")
    
    # Set up stealth settings for standard Selenium
    options = Options()
    # 1. Remove the "Chrome is being controlled by automated test software" banner
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    # 2. Mask the automation flags from the website's firewall
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    # Launch the browser
    driver = webdriver.Chrome(options=options)
    
    # Execute a small script to further mask the webdriver signature
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    driver.get(main_url)
    
    # 🛑 Give the browser 5 seconds to load and pass any Cloudflare checks
    time.sleep(5) 
    
    # Grab the HTML source code from the live browser
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    card_links = soup.find_all('a', class_=re.compile(r'wrap-link'))
    
    faculty_list = []

    for card in card_links:
        profile_path = card.get('href')
        if not profile_path:
            continue
            
        name_tag = card.find('h2', class_=re.compile(r'title'))
        name = name_tag.get_text(strip=True) if name_tag else "N/A"
        
        if name == "N/A" or not name:
            continue
            
        content_div = card.find('div', class_='block-content')
        if content_div:
            p_tag = content_div.find('p')
            if p_tag and p_tag.find('strong'):
                position = p_tag.find('strong').get_text(strip=True)
            else:
                position = "N/A"
        else:
            position = "N/A"

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} faculty members. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            # Navigate the live browser to the profile page
            driver.get(person['Profile_URL'])
            time.sleep(1.5) # Wait for page to render
            
            prof_soup = BeautifulSoup(driver.page_source, 'html.parser')
            email_tag = prof_soup.find('a', href=re.compile(r'^mailto:'))
            
            if email_tag:
                raw_email = email_tag['href'].replace('mailto:', '')
                person['Email'] = raw_email.split('?')[0].strip()
            else:
                person['Email'] = "N/A"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })

    # Close the browser when finished
    driver.quit()
    return results

# Execute the Crawler
data = crawl_haas_stealth_native()

if data:
    filename = 'berkeley_haas_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Initializing stealth browser...
Phase 1: Fetching main directory... (https://haas.berkeley.edu/faculty/)
Found 35 faculty members. Starting Phase 2...

Scraping 1/35: David A. Aaker...
Scraping 2/35: Kai Adams...
Scraping 3/35: Mark Adams...
Scraping 4/35: Vinod K Aggarwal...
Scraping 5/35: Mario Alvarez...
Scraping 6/35: Satish Ananthaswamy...
Scraping 7/35: Cameron Anderson...
Scraping 8/35: Maximilian Auffhammer...
Scraping 9/35: Ned Augenblick...
Scraping 10/35: Wasim Azhar...
Scraping 11/35: Matthew Backus...
Scraping 12/35: Ahmed Badruzzaman...
Scraping 13/35: Roy Bahat...
Scraping 14/35: Homa Bahrami...
Scraping 15/35: Erica R. Bailey...
Scraping 16/35: Rajiv Ball...
Scraping 17/35: Cristina G. Banks...
Scraping 18/35: Sara L. Beckman...
Scraping 19/35: Erick O. Bell...
Scraping 20/35: Matteo Benetton...
Scraping 21/35: Martin Beraja...
Scraping 22/35: Peter Bershatsky...
Scraping 23/35: Kurt Beyer...
Scraping 24/35: Matilde Bombardini...
Scraping 25/35: Olivia Bordeu...
Scrapin

In [18]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_open_berkeley():
    # If the URL is different, just swap it out here
    base_url = "https://math.berkeley.edu" 
    main_url = f"{base_url}/people/faculty"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 🎯 TARGET 1: The overarching profile card container
    # From screenshot 1: <div class="node node-openberkeley-person node-view--card row">
    cards = soup.find_all('div', class_=re.compile(r'node-openberkeley-person'))
    
    faculty_list = []

    for card in cards:
        # The main wrapper is an <a> tag right inside the content div
        link_tag = card.find('a', href=True)
        if not link_tag:
            continue
            
        profile_path = link_tag['href']
        
        # 1. Get the Name (Inside an <h2> tag)
        name_tag = card.find('h2')
        name = name_tag.get_text(strip=True) if name_tag else "N/A"
        
        # Skip if no name found
        if name == "N/A" or not name:
            continue
            
        # 2. Get the Position 
        # Targeting <div class="field-name-field-openberkeley-person-title">
        title_div = card.find('div', class_=re.compile(r'field-name-field-openberkeley-person-title'))
        if title_div:
            # The actual text is inside a 'field-item' div
            item_div = title_div.find('div', class_=re.compile(r'field-item'))
            position = item_div.get_text(strip=True) if item_div else "N/A"
        else:
            position = "N/A"

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} profiles. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # 🎯 TARGET 2: The exact email field wrapper
                # From screenshot 2: <div class="field-name-field-openberkeley-person-email">
                email_wrapper = prof_soup.find('div', class_=re.compile(r'field-name-field-openberkeley-person-email'))
                
                if email_wrapper:
                    # Find the mailto link inside that wrapper
                    email_tag = email_wrapper.find('a', href=re.compile(r'^mailto:'))
                    if email_tag:
                        raw_email = email_tag['href'].replace('mailto:', '')
                        person['Email'] = raw_email.split('?')[0].strip()
                    else:
                        person['Email'] = "N/A"
                else:
                    person['Email'] = "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # Politeness delay
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_open_berkeley()

if data:
    filename = 'berkeley_open_template_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://math.berkeley.edu/people/faculty)
Found 146 profiles. Starting Phase 2...

Scraping 1/146: Mina Aganagic...
Scraping 2/146: Ian Agol...
Scraping 3/146: David Aldous...
Scraping 4/146: Robert M. Anderson...
Scraping 5/146: Richard Bamler...
Scraping 6/146: Hélène Barcelo...
Scraping 7/146: Khalilah Beal...
Scraping 8/146: George M. Bergman...
Scraping 9/146: Adam Black...
Scraping 10/146: Richard E. Borcherds...
Scraping 11/146: Felix Brandt...
Scraping 12/146: Robert Bryant...
Scraping 13/146: Sunčica Čanić...
Scraping 14/146: Jennifer Chayes...
Scraping 15/146: Alexandre J. Chorin...
Scraping 16/146: Michael Christ...
Scraping 17/146: Paul Concus...
Scraping 18/146: Sylvie Corteel...
Scraping 19/146: James W. Demmel...
Scraping 20/146: David Eisenbud...
Scraping 21/146: Ethan N. Epperly...
Scraping 22/146: L. Craig Evans...
Scraping 23/146: Steven N. Evans...
Scraping 24/146: Tony Feng...
Scraping 25/146: Edward Frenkel...
Scraping 26/146: 

In [19]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_psychology_faculty():
    # Updated to the Psychology department URL
    base_url = "https://psychology.berkeley.edu" 
    
    # If the main list is on a different page, change this URL 
    main_url = f"{base_url}/people/faculty"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Targeting the standardized Open Berkeley profile card
    cards = soup.find_all('div', class_=re.compile(r'node-openberkeley-person'))
    
    faculty_list = []

    for card in cards:
        link_tag = card.find('a', href=True)
        if not link_tag:
            continue
            
        profile_path = link_tag['href']
        
        # 1. Get Name
        name_tag = card.find('h2')
        name = name_tag.get_text(strip=True) if name_tag else "N/A"
        
        if name == "N/A" or not name:
            continue
            
        # 2. Get Position
        title_div = card.find('div', class_=re.compile(r'field-name-field-openberkeley-person-title'))
        if title_div:
            item_div = title_div.find('div', class_=re.compile(r'field-item'))
            position = item_div.get_text(strip=True) if item_div else "N/A"
        else:
            position = "N/A"

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} profiles. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # Targeting the standardized Open Berkeley email wrapper
                email_wrapper = prof_soup.find('div', class_=re.compile(r'field-name-field-openberkeley-person-email'))
                
                if email_wrapper:
                    email_tag = email_wrapper.find('a', href=re.compile(r'^mailto:'))
                    if email_tag:
                        raw_email = email_tag['href'].replace('mailto:', '')
                        person['Email'] = raw_email.split('?')[0].strip()
                    else:
                        person['Email'] = "N/A"
                else:
                    person['Email'] = "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # Politeness delay
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_psychology_faculty()

if data:
    filename = 'berkeley_psychology_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://psychology.berkeley.edu/people/faculty)
Found 34 profiles. Starting Phase 2...

Scraping 1/34: Mariam Aly...
Scraping 2/34: Ozlem Ayduk...
Scraping 3/34: Jasmin Brooks Stephens...
Scraping 4/34: Silvia Bunge...
Scraping 5/34: Serena Chen...
Scraping 6/34: Anne Collins...
Scraping 7/34: Mark T. D'Esposito...
Scraping 8/34: Gul Dolen...
Scraping 9/34: Arianne Eason...
Scraping 10/34: Jan Engelmann...
Scraping 11/34: Aaron Fisher...
Scraping 12/34: Allison Harvey...
Scraping 13/34: Stephen Hinshaw...
Scraping 14/34: Richard Ivry...
Scraping 15/34: Oliver P. John...
Scraping 16/34: Sheri Johnson...
Scraping 17/34: Keanan Joyner...
Scraping 18/34: Dacher Keltner...
Scraping 19/34: Celeste Kidd...
Scraping 20/34: Hedy Kober...
Scraping 21/34: Lance Kriegsfeld...
Scraping 22/34: Nancy Liu...
Scraping 23/34: Iris Mauss...
Scraping 24/34: R. Mendoza-Denton...
Scraping 25/34: Steven Piantadosi...
Scraping 26/34: Giovanni Ramos...
Scraping 27/34: Amita

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_ieor_precise():
    base_url = "https://ieor.berkeley.edu"
    main_url = f"{base_url}/faculty/" # Change to /faculty/ if that is the main directory
    
    print("Initializing stealth browser...")
    
    # Set up stealth settings for standard Selenium
    options = Options()
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
    
    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    driver.get(main_url)
    
    # 🛑 CRITICAL DELAY: 5 seconds to pass any firewall checks
    time.sleep(5) 
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # 🎯 TARGET 1: Strict filtering. Only find H2 tags with the title class.
    title_tags = soup.find_all('h2', class_=re.compile(r'fl-callout-title'))
    
    faculty_list = []

    for title in title_tags:
        # Find the <a> tag inside the H2
        link_tag = title.find('a', href=True)
        if not link_tag:
            continue
            
        href = link_tag['href']
        
        # 🎯 THE FILTER: Only process links that actually go to a person's page
        if '/people/' not in href and '/faculty/' not in href:
            continue
            
        name = link_tag.get_text(strip=True)
        
        # The position is in the div that comes immediately after the H2
        text_wrap = title.find_next_sibling('div', class_=re.compile(r'fl-callout-text-wrap'))
        position = text_wrap.get_text(separator=", ", strip=True) if text_wrap else "N/A"

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, href),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} actual profiles. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive (With Anti-Obfuscation) ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            # Navigate to Ilan Adler (or whoever's) specific page
            driver.get(person['Profile_URL'])
            time.sleep(1.5) 
            
            prof_soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            # Grab the text from the accordion dropdown, or the whole page if missing
            accordion_div = prof_soup.find('div', class_=re.compile(r'fl-accordion-content'))
            search_area = accordion_div.get_text(" ", strip=True) if accordion_div else prof_soup.get_text(" ", strip=True)
            
            # 🎯 TARGET 2: Advanced Regex to catch standard emails AND "ilan(at)berkeley.edu"
            email_match = re.search(r'([a-zA-Z0-9._%+-]+(?:@|\(at\)|\[at\]|\s@\s)[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', search_area, re.IGNORECASE)
            
            if email_match:
                raw_email = email_match.group(1)
                # Translate the spelled out (at) back into an actual @ symbol
                clean_email = re.sub(r'\(at\)|\[at\]|\s@\s', '@', raw_email)
                person['Email'] = clean_email
            else:
                person['Email'] = "N/A"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })

    # Close the stealth browser
    driver.quit()
    return results

# Execute the Crawler
data = crawl_ieor_precise()

if data:
    filename = 'berkeley_ieor_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Initializing stealth browser...
Phase 1: Fetching main directory... (https://ieor.berkeley.edu/faculty/)
Found 30 actual profiles. Starting Phase 2...

Scraping 1/30: Ilan Adler...
Scraping 2/30: Anil Aswani...
Scraping 3/30: Alper Atamturk...
Scraping 4/30: Ying Cui...
Scraping 5/30: Lee Fleming...
Scraping 6/30: Ken Goldberg...
Scraping 7/30: Paul Grigas...
Scraping 8/30: Xin Guo...
Scraping 9/30: Dorit Hochbaum...
Scraping 10/30: Huiwen Jia...
Scraping 11/30: Philip M. Kaminsky...
Scraping 12/30: Phillip Kerger...
Scraping 13/30: Javad Lavaei...
Scraping 14/30: Robert Leachman...
Scraping 15/30: Thibaut Mastrolia...
Scraping 16/30: Shmuel Oren...
Scraping 17/30: Daniel Pirutinsky...
Scraping 18/30: Rhonda Righter...
Scraping 19/30: Rajan Udwani...
Scraping 20/30: Chiwei Yan...
Scraping 21/30: Candace Yano...
Scraping 22/30: Zeyu Zheng...
Scraping 23/30: Richard E. Barlow...
Scraping 24/30: Stuart Dreyfus...
Scraping 25/30: Robert M. Oliver...
Scraping 26/30: Sheldon Ross...
Scraping

In [13]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv

def scrape_anthropology_faculty_fixed():
    url = "https://anthropology.berkeley.edu/people/faculty-k-z"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Fetching directory... ({url})")
    response = scraper.get(url)
    
    if response.status_code != 200:
        print(f"Failed to load page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    h2_tags = soup.find_all('h2')
    results = []

    for h2 in h2_tags:
        # 1. Get the Name
        name_tag = h2.find('a')
        if not name_tag:
            continue
            
        name = name_tag.get_text(strip=True)
        if len(name) < 2:
            continue
            
        # 2. Get the Position (FIXED LOGIC)
        pos_parts = []
        
        # Check Location A: Title stuffed inside the H2 (e.g., "Department Chair")
        # We grab the full H2 text, erase the Name, and see if anything is left over
        extra_h2_text = h2.get_text(strip=True).replace(name, '').strip(', \xa0|')
        if extra_h2_text:
            pos_parts.append(extra_h2_text)
            
        # Check Location B: Title in the <p> tag immediately following the H2
        next_p = h2.find_next_sibling('p')
        if next_p:
            strong_tag = next_p.find('strong')
            if strong_tag:
                # Get the text and replace HTML non-breaking spaces (\xa0) with normal spaces
                p_text = strong_tag.get_text(strip=True).replace('\xa0', ' ')
                pos_parts.append(p_text)
        
        # Combine them if both exist, otherwise default to "Faculty"
        position = " | ".join(pos_parts) if pos_parts else "Faculty"

        # 3. Get the Email
        parent_div = h2.parent
        email = "N/A"
        
        if parent_div:
            email_tag = parent_div.find('a', href=re.compile(r'^mailto:'))
            if email_tag:
                raw_email = email_tag['href'].replace('mailto:', '')
                email = raw_email.split('?')[0].strip()

        results.append({
            'Name': name,
            'Position': position,
            'Email': email
        })

    return results

# Execute the Scraper
data = scrape_anthropology_faculty_fixed()

if data:
    filename = 'berkeley_anthropology_faculty2.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Extracted {len(data)} profiles. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Fetching directory... (https://anthropology.berkeley.edu/people/faculty-k-z)

✅ Success! Extracted 15 profiles. Saved to berkeley_anthropology_faculty2.csv.

--- Preview ---
{'Name': 'Andrew Wooyoung Kim', 'Position': 'Assistant Professor |  Biological Anthropology', 'Email': 'awkim@berkeley.edu'}
{'Name': 'Nicholas Laluk', 'Position': 'Assistant Professor | Archaeology', 'Email': 'nlaluk@berkeley.edu'}
{'Name': 'Xin Liu', 'Position': 'Professor  |  Sociocultural Anthropology', 'Email': 'xinliu@berkeley.edu'}


In [15]:
import cloudscraper
from bs4 import BeautifulSoup
import re
import csv
import time
from urllib.parse import urljoin

def crawl_open_berkeley_standard():
    # 🛑 INSTRUCTION: Paste the department's main homepage URL here
    base_url = "https://tdps.berkeley.edu" 
    
    # 🛑 INSTRUCTION: Adjust if their directory is at /people instead of /people/faculty
    main_url = f"{base_url}/people/faculty"
    
    scraper = cloudscraper.create_scraper(browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    })
    
    print(f"Phase 1: Fetching main directory... ({main_url})")
    response = scraper.get(main_url)
    
    if response.status_code != 200:
        print(f"Failed to load main page. Status: {response.status_code}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Targeting the standardized Open Berkeley profile card
    cards = soup.find_all('div', class_=re.compile(r'node-openberkeley-person'))
    
    faculty_list = []

    for card in cards:
        # Find the h2 tag which holds the link and name
        h2_tag = card.find('h2')
        if not h2_tag:
            continue
            
        link_tag = h2_tag.find('a', href=True)
        if not link_tag:
            continue
            
        profile_path = link_tag['href']
        name = link_tag.get_text(strip=True)
        
        if len(name) < 2:
            continue
            
        # Get Position
        title_div = card.find('div', class_=re.compile(r'field-name-field-openberkeley-person-title'))
        if title_div:
            item_div = title_div.find('div', class_=re.compile(r'field-item'))
            position = item_div.get_text(strip=True) if item_div else "N/A"
        else:
            position = "N/A"

        faculty_list.append({
            'Name': name,
            'Position': position,
            'Profile_URL': urljoin(base_url, profile_path),
            'Email': 'Pending...'
        })

    print(f"Found {len(faculty_list)} profiles. Starting Phase 2...\n")
    
    # --- PHASE 2: The Deep Dive ---
    results = []
    
    for i, person in enumerate(faculty_list, 1):
        print(f"Scraping {i}/{len(faculty_list)}: {person['Name']}...")
        
        try:
            prof_response = scraper.get(person['Profile_URL'])
            
            if prof_response.status_code == 200:
                prof_soup = BeautifulSoup(prof_response.text, 'html.parser')
                
                # Targeting the standardized Open Berkeley email wrapper
                email_wrapper = prof_soup.find('div', class_=re.compile(r'field-name-field-openberkeley-person-email'))
                
                if email_wrapper:
                    email_tag = email_wrapper.find('a', href=re.compile(r'^mailto:'))
                    if email_tag:
                        raw_email = email_tag['href'].replace('mailto:', '')
                        person['Email'] = raw_email.split('?')[0].strip()
                    else:
                        person['Email'] = "N/A"
                else:
                    person['Email'] = "N/A"
            else:
                person['Email'] = "Page Error"
                
        except Exception as e:
            person['Email'] = "Error Fetching"
            
        results.append({
            'Name': person['Name'],
            'Position': person['Position'],
            'Email': person['Email']
        })
        
        # Politeness delay
        time.sleep(1)

    return results

# Execute the Crawler
data = crawl_open_berkeley_standard()

if data:
    filename = 'berkeley_department_faculty.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Name', 'Position', 'Email'])
        writer.writeheader()
        writer.writerows(data)
    
    print(f"\n✅ Success! Scrape complete. Saved to {filename}.")
    print("\n--- Preview ---")
    for row in data[:3]:
        print(row)

Phase 1: Fetching main directory... (https://tdps.berkeley.edu/people/faculty)
Found 33 profiles. Starting Phase 2...

Scraping 1/33: Brandi Wilkins Catanese...
Scraping 2/33: Abigail De Kosnik...
Scraping 3/33: Timmia Hearn DeRoy...
Scraping 4/33: Julia Fawcett...
Scraping 5/33: Karina Gutiérrez...
Scraping 6/33: Shannon Jackson...
Scraping 7/33: Roshanak Kheshti...
Scraping 8/33: SanSan Kwan...
Scraping 9/33: Angela Marino...
Scraping 10/33: Shannon Steen...
Scraping 11/33: Lisa Wymore...
Scraping 12/33: Sima Belmar...
Scraping 13/33: Nancy Carlin...
Scraping 14/33: Iu-Hui Chua...
Scraping 15/33: Rebecca J. Ennals...
Scraping 16/33: Srijani Ghosh...
Scraping 17/33: Bear Graham...
Scraping 18/33: Chelsea Gregory...
Scraping 19/33: Margo Hall...
Scraping 20/33: Christopher Herold...
Scraping 21/33: Dominique Fawn Hill...
Scraping 22/33: Jessica Berman Hirigoyen...
Scraping 23/33: Laxmi Kumaran...
Scraping 24/33: Daniel Larlham...
Scraping 25/33: Erik K. Raymond Lee...
Scraping 26/33: S